# MP20 — SRCNN Implementation

SRCNN first upsamples the input with bicubic interpolation, then learns feature extraction, nonlinear mapping and reconstruction at the HR grid. This lesson uses a compact adaptation, not the original paper architecture or training schedule.

## Mathematics, shapes and worked example

Let y be LR and B(y) its bicubic x2 image. F1=ReLU(W1*B(y)+b1), F2=ReLU(W2*F1+b2), output=W3*F2+b3. The asterisk denotes library cross-correlation. Kernels are 9×9, 1×1 and 5×5 with widths 32 and 16. All convolutions are padded, unlike an exact original valid-convolution reproduction. Input 1×3×16×16 becomes 1×3×32×32. The 1×1 mapping changes channels but not spatial dimensions.

In [ ]:
from pathlib import Path
import sys,json,torch
ROOT=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'src/rssr').is_dir())
sys.path.insert(0,str(ROOT/'src'))
torch.manual_seed(19)
from mp20_tools import SRModel
model=SRModel(kind='srcnn',bands=3)
x=torch.rand(1,3,16,16)
assert model(x).shape==(1,3,32,32)
print(model.srcnn)
print('Parameters',sum(p.numel() for p in model.parameters()))

## Reproducible experiment

Run `python scripts/cnn_lessons_experiment.py` from the repository root. The script first checks a small synthetic input/output shape. It then uses four training and four validation NAIP scenes, split before tiling: 16 nonoverlapping 32×32 RGB patches per scene. HR stored DN are divided by 10000, not claimed as reflectance. SR inputs use Gaussian sigma 0.8 HR pixels and area x2 downsampling. Identity reconstruction uses HR as both input and target.

Training: CPU float32, two threads, seed 19, Adam learning rate 0.001, batch four, 300 updates, MSE, no augmentation or scheduler. Evaluate the final fixed step; no validation-based checkpoint search. Teaching weights are regenerated and not committed. The constant baseline uses training-channel means. SR baselines use bicubic with align_corners=False. Metrics use range one, border two, all RGB channels; equal patches per scene give equal scene weighting.

Data: existing checksummed NAIP/OpenSR crops, public domain with USDA/OpenSR attribution. No new imagery download. Final research weights/protocol/test results are untouched by these separate teaching exercises. No learner mastery claim.

## Observed result and interpretation

The 300-step compact SRCNN scored 37.737 dB on validation versus bicubic 42.964 dB. Preserve this negative result: a trainable architecture does not guarantee a better image after a tiny budget. This lesson reports validation only; its model was not added to the already frozen research test. The brief’s independent held-out SRCNN table remains unfulfilled, rather than relabeling validation as test.

In [ ]:
report=json.loads((ROOT/'reports/mp19-mp22-cnn.json').read_text())
selected=['srcnn_mse']
for row in report['results']:
 if row['model'] in selected:
  print({k:v for k,v in row.items() if k!='per_tile'})
from IPython.display import Image,display
display(Image(filename=str(ROOT/'reports/mp19-mp22-cnn.png')))

## Practice — solve first

1. Why does this architecture work mainly on the HR grid?
2. Does a 1×1 convolution mix neighboring pixels?
3. Can its validation score be called a paper reproduction?

## Worked solutions

1. Bicubic upsampling happens before learned feature extraction, increasing spatial computation.
2. No; it mixes channels independently at each pixel.
3. No. Widths, padding, data, degradation and training budget differ.

## Limitations and next question

One seed and a short budget describe an engineering demonstration. Interpolation, padding, input distribution and unequal parameter counts affect results. Investigate these factors on validation before any new frozen research test. Generated answers do not assess learner mastery.

[Authoritative brief](README.md) · [Local/Colab setup](../../README.md)